# Embedding-Split.py

This notebook is used to split the existing embeddings files, or a subset of them, into chucked up files, making it easier to ingest them into other environments and OpenSearch in AWS.

## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [1]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec

Note: you may need to restart the kernel to use updated packages.


Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [2]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(
    vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential
)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = "workspaceblobstore"

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [3]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [4]:
import torch

USE_EXACT_SEARCH = False

if USE_EXACT_SEARCH:
    assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), while embedding tensor files and any HNSW `.index` files do not, and can simply be loaded directly from storage.

In [5]:
# Load up the validation set data
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

/anaconda/envs/dibbs_env/lib/python3.10/site-packages/azureml/dataprep/api/_loggerfactory.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Step 2: Unpickle Embeddings Files and then splitting them up into JSONL files

In [6]:
import json
import os
import pickle

CHUNK_SIZE = 1000


print("START PROCESSING FILES....")
# loop through each embedding file
for file in fs.ls("embeddings/refined"):
    # skip any sub-folders
    if not fs.isfile(file) or file not in (
        "embeddings/refined/loinc_lab_names_Snowflake_snowflake-arctic-embed-l-v2.0_0.3_1e05_20251008",
        "embeddings/refined/loinc_lab_names_Snowflake_snowflake-arctic-embed-l-v2.0_0.3_1e06_20251008",
        "embeddings/refined/loinc_lab_names_Snowflake_snowflake-arctic-embed-l-v2.0_0.3_5e05_20251008",
        "embeddings/refined/loinc_lab_names_Snowflake_snowflake-arctic-embed-l-v2.0_0.3_5e06_20251008",
        "embeddings/refined/loinc_lab_names_Snowflake_snowflake-arctic-embed-l-v2.0_20251008",
        "embeddings/refined/loinc_lab_names_BAAI_bge-base-en-v1.5_20251008",
        "embeddings/refined/loinc_lab_names_BAAI_bge-large-en-v1.5_20251008",
        "embeddings/refined/loinc_lab_names_intfloat_e5-base-v2_20251008",
        "embeddings/refined/loinc_lab_names_intfloat_e5-large-v2_20251008",
        "embeddings/refined/loinc_lab_names_Qwen_Qwen3-Embedding-4B_20251008",
    ):
        continue
    print(f"OPEN FILE: {file}")
    # open the embedding file and un-pickle it
    # store the elements in arrays
    with fs.open(file) as fp:
        print("UN-PICKLE")
        cache_data = pickle.load(fp)
        name_codes = cache_data["codes"]
        embeddings = cache_data["embeddings"]
        loinc_types = cache_data["loinc_types"]

        print("SIZE:")
        print(len(name_codes))
        print(len(loinc_types))

        # Convert to numpy if tensor
        if isinstance(embeddings, torch.Tensor):
            print("Converting embeddings from torch.Tensor to numpy array")
            embeddings = embeddings.cpu().numpy()

    print("WRITE LOCAL FILES...")
    # use the embedding file name to be used to
    # store a local update embedding file which will
    # later be 'put' into the embeddings folder in Azure
    local_file = file.split("/", 2)[2]
    output_folder = os.path.join(os.getcwd(), "split-embeddings", local_file)
    os.makedirs(output_folder, exist_ok=True)
    print(f"Converting model {local_file}")

    for i in range(0, len(name_codes), CHUNK_SIZE):
        chunk = [
            {
                "id": str(i + j),
                "description": name_codes[i + j],
                "descriptionVector": embeddings[i + j].tolist(),
                "type": loinc_types[i + j],
            }
            for j in range(min(CHUNK_SIZE, len(name_codes) - i))
        ]
        with open(f"{output_folder}/{local_file}_{i // CHUNK_SIZE:05d}.jsonl", "w") as f:
            f.writelines(json.dumps(doc) + "\n" for doc in chunk)

        print(f"Wrote chunk {i // CHUNK_SIZE}")
print("UPLOADING FOLDER TO BLOB STORAGE...")
input_folder = os.path.join(os.getcwd(), "split-embeddings")
fs.upload(lpath=input_folder, rpath="/embeddings/refined/split/", recursive=True)
os.rmdir(input_folder)

print("PROCESS DONE!")

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


OPEN FILE: embeddings/refined/loinc_lab_names_BAAI_bge-base-en-v1.5_20251008
UN-PICKLE
SIZE:
242925
242925
Converting embeddings from torch.Tensor to numpy array
WRITE LOCAL FILES...
Converting model loinc_lab_names_BAAI_bge-base-en-v1.5_20251008
Wrote chunk 0
Wrote chunk 1
Wrote chunk 2
Wrote chunk 3
Wrote chunk 4
Wrote chunk 5
Wrote chunk 6
Wrote chunk 7
Wrote chunk 8
Wrote chunk 9
Wrote chunk 10
Wrote chunk 11
Wrote chunk 12
Wrote chunk 13
Wrote chunk 14
Wrote chunk 15
Wrote chunk 16
Wrote chunk 17
Wrote chunk 18
Wrote chunk 19
Wrote chunk 20
Wrote chunk 21
Wrote chunk 22
Wrote chunk 23
Wrote chunk 24
Wrote chunk 25
Wrote chunk 26
Wrote chunk 27
Wrote chunk 28
Wrote chunk 29
Wrote chunk 30
Wrote chunk 31
Wrote chunk 32
Wrote chunk 33
Wrote chunk 34
Wrote chunk 35
Wrote chunk 36
Wrote chunk 37
Wrote chunk 38
Wrote chunk 39
Wrote chunk 40
Wrote chunk 41
Wrote chunk 42
Wrote chunk 43
Wrote chunk 44
Wrote chunk 45
Wrote chunk 46
Wrote chunk 47
Wrote chunk 48
Wrote chunk 49
Wrote chunk 5